## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

Examples:

- Summarization	
- Human-in-the-loop	
- Model call limit	
- Tool call limit
- Model fallback
- PII detection
- To-do list

In [13]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model_name = "groq:qwen/qwen3-32b"

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

### Message based summarization

In [6]:
from langchain.agents import create_agent 
from langchain.agents.middleware import SummarizationMiddleware 
#from langchain.checkpoint.memory import InMemorySaver 
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.messages import HumanMessage, SystemMessage 

# message based summarization
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model = "groq:qwen/qwen3-32b", # use small model 
        trigger = ("messages", 10), # when message reched 10 then trigger the summarization 
        keep = ("messages", 4) # keep recent 4 messages
    )]
)

In [7]:
# run with thread id

config = {"configurable": {"thread_id": "test-1"}}

In [8]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content = q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='7df8bcfe-4c6e-4138-a2b6-c01d96ae2045'), AIMessage(content='<think>\nOkay, so the user is asking, "What is 2+2?" Hmm, that\'s a basic arithmetic question. Let me think through this step by step. First, I need to recall the fundamental principles of addition. Addition is one of the four basic operations in arithmetic, along with subtraction, multiplication, and division. When you add two numbers, you\'re essentially combining their quantities.\n\nStarting with the number 2, if I add another 2, I should count forward by 2 from the original 2. So, 2 plus 1 is 3, and then adding another 1 (since 2 is 1+1) would make it 4. Alternatively, using the number line concept, starting at 2 and moving 2 units to the right lands me at 4. \n\nAnother way to approach this is through the concept of place value. In the decimal system, each digit represents a value based on its position. Here, both 

### Based on Token size summarization

In [10]:
from langchain.tools import tool 

@tool 
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""
    Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [SummarizationMiddleware(
        model = "groq:qwen/qwen3-32b",
        trigger = ("tokens", 500),
        keep = ("tokens", 200)
    )]
)

config = {"configurable": {"thread_id": "test-2"}}

# token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [11]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content = f"Find hotels in {city}")]},
        config = config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~136 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='d8619484-f5f8-4a64-8d01-e1d46d680757'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be {"city": "Paris"}. Since the function is designed to return a long response, I should expect that once I get the data, I\'ll need to present it in a detailed way. But first, just need to make the tool call with the right arguments. No other parameters are required, so it\'s straightforward. Let me format the tool call as specified.\n', 'tool_calls': [{'id': 'y2b6yf59a', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_toke

### Based on Fraction

In [12]:
# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~123 tokens (0.0961%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='01932bdb-ae8e-471a-ad57-7d32a96dc6b0'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. Since the user mentioned Paris, I need to call that function with "Paris" as the city. The function returns a long response, so I should make sure to use it properly. I\'ll generate the tool call with the city parameter set to Paris. No other parameters are needed because the function only requires the city. Let me double-check that the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'fs352qr3n', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 155, 'total_t

## Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [26]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model = model_name,
    checkpointer = InMemorySaver(),
    tools=[read_email_tool,send_email_tool],
    middleware = [HumanInTheLoopMiddleware(
        interrupt_on = {
            "send_email_tool": {
                "allowed_decisions": ["approve","edit","reject"]
            },
            "read_email_tool": False
        }
    )]
)

In [27]:
config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config = config 
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f3d64923-40eb-4876-81c5-f9297c3027db'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. Just make sure the JSON is correctly formatted with the parameters.\n", 'tool_calls': [{'id': 'xyfpfv15b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'comple

#### Approve

In [28]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={"decisions": [{"type": "approve"}]}
        ),
        config = config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello".


In [29]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='f3d64923-40eb-4876-81c5-f9297c3027db'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. Just make sure the JSON is correctly formatted with the parameters.\n", 'tool_calls': [{'id': 'xyfpfv15b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'comple

#### Reject

In [31]:

config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

result

⏸️ Paused! Approving...
✅ Result: The tool call to send the email was rejected. If you'd like me to retry or take another action, please let me know explicitly.


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='21436c0e-91cf-4cd1-80e9-0da180d1aed2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. I need to check which tool to use here. The available tools are read_email_tool and send_email_tool. Since the action is sending an email, the send_email_tool is the right choice. The parameters required for send_email_tool are recipient, subject, and body. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. I should structure the arguments into a JSON object with these values. Make sure the keys are correct and the values are strings. No need to use the read_email_tool here because the user isn't asking to read an email by ID. Just call send_email_tool with th

#### Editing

In [32]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

result

⏸️ Paused! Editing...
✏️ Result: The email has been successfully sent to **correct@email.com** with the subject **"Corrected Subject"** and the body **"This was edited by human before sending"**. Let me know if you'd like to review or adjust the original draft!


{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='1600dfe4-c69f-43b9-beb4-044fa9f6b757'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yes, they are. So I should call the send_email_tool with these arguments. No need to use the read_email_tool here since the task is about sending, not reading. Everything looks set. Let's structure the tool call accordingly.\n", 'tool_calls': [{'id': 'an9cm24wc', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_ema